# Fase 1 — Nuevas características léxicas

4 características nuevas, integradas en `logic/feature_extraction.py` y `logic/lexical_features.py`, cada una con hipótesis lingüística, implementación vectorizable y prueba con tweets reales — cumpliendo el mínimo exigido (4, para grupo de 2 integrantes) y verificando que ninguna duplique las ya existentes ni caiga en los problemas detectados en la Fase 0 (constante / proxy de longitud).

| # | Característica | Reemplaza / se relaciona con | Motivo |
|---|---|---|---|
| 1 | `lexical_diversity_mattr` | `lexical_diversity` (excluida en Fase 0) | TTR corregido por ventana móvil, no depende de la longitud |
| 2 | `elongated_words_count` | — (nueva) | Elongación de caracteres ("buenooo") |
| 3 | `punct_repetition` | — (nueva) | Repetición de `!`/`?` |
| 4 | `lex_pol_neg` | `adjetives_pos`/`adjetives_neg` (existentes, se mantienen) | Polaridad con alcance de negación — NO duplica, calcula un mecanismo distinto (puntaje neto con negación resuelta, no conteo crudo) |


In [1]:
import sys
sys.path.insert(0, '..')

import importlib
import logic.lexical_features, logic.text_processing, logic.feature_extraction
importlib.reload(logic.lexical_features)
importlib.reload(logic.text_processing)
importlib.reload(logic.feature_extraction)

from logic.feature_extraction import FeatureExtraction
from logic.text_processing import TextProcessing
from nltk import TweetTokenizer
import pandas as pd
import numpy as np

fe = FeatureExtraction(lang='es')
tt = TweetTokenizer()
df = pd.read_csv('../data/tass/tass2018_es_train.csv')
texts = df['content'].dropna().tolist()
print(f"N tweets: {len(texts)}")


N tweets: 1008


## 1. `lex_pol_neg` — polaridad léxica con alcance de negación

**Hipótesis:** un conteo simple de palabras positivas/negativas falla cuando la palabra está bajo el alcance de una negación (`"no me gusta"` cuenta `"gusta"` como positivo, siendo el enunciado negativo). Esta característica invierte el signo de las palabras del léxico que caen dentro del alcance de un marcador de negación, hasta el siguiente límite (conjunción o fin del tweet).

**Por qué no duplica `adjetives_pos`/`adjetives_neg`:** esas features cuentan ocurrencias crudas sin resolver negación; esta calcula un puntaje neto con la negación ya resuelta — es un mecanismo distinto, no una repetición.


In [2]:
casos_negacion = [
    ("no me gusta nada esto", "negativo (negacion invierte 'gusta')"),
    ("me encanta esto", "positivo (sin negacion)"),
    ("no me gusta pero es bonito", "'pero' corta el alcance de la negacion"),
]

for texto, esperado in casos_negacion:
    proc = TextProcessing.transformer(texto)
    tokens = tt.tokenize(proc)
    score = FeatureExtraction.lex_pol_neg(tokens)
    print(f"{texto!r:45s} -> score={score:+.1f}  (se espera: {esperado})")


'no me gusta nada esto'                       -> score=-1.0  (se espera: negativo (negacion invierte 'gusta'))
'me encanta esto'                             -> score=+1.0  (se espera: positivo (sin negacion))
'no me gusta pero es bonito'                  -> score=+0.0  (se espera: 'pero' corta el alcance de la negacion)


## 2. `elongated_words_count` — elongación de caracteres

In [3]:
casos_elongacion = ["buenooo dia", "hola amigo", "nooooo puede ser", "jajaja que risa"]
for c in casos_elongacion:
    print(f"{c!r:30s} -> {FeatureExtraction.elongated_words_count(c)}")


'buenooo dia'                  -> 1.0
'hola amigo'                   -> 0.0
'nooooo puede ser'             -> 1.0
'jajaja que risa'              -> 0.0


## 3. `punct_repetition` — repetición de signos de exclamación/interrogación

In [4]:
casos_puntuacion = ["que bien!!!", "en serio???", "hola.", "genial!! de verdad??"]
for c in casos_puntuacion:
    print(f"{c!r:30s} -> {FeatureExtraction.punct_repetition(c)}")


'que bien!!!'                  -> 1.0
'en serio???'                  -> 1.0
'hola.'                        -> 0.0
'genial!! de verdad??'         -> 2.0


## 4. `lexical_diversity_mattr` — diversidad léxica corregida por longitud

**Hipótesis:** la `lexical_diversity` original está confundida con la longitud del texto (r=-0.83, según la auditoría de la Fase 0) y, además, mide diversidad de *caracteres*, no de *palabras* (error de implementación detectado por separado). MATTR promedia el TTR sobre ventanas móviles de tamaño fijo, corrigiendo el sesgo de longitud.

Verificación empírica sobre 300 tweets reales:


In [5]:
mattr_vals, n_tok_vals, old_ld_vals = [], [], []
for t in texts[:300]:
    proc = TextProcessing.transformer(t)
    if not proc:
        continue
    tokens = tt.tokenize(proc)
    if len(tokens) == 0:
        continue
    mattr_vals.append(FeatureExtraction.lexical_diversity_mattr(tokens, window=10))
    n_tok_vals.append(len(tokens))
    old_ld_vals.append(fe.lexical_diversity(proc))

corr_mattr = np.corrcoef(mattr_vals, n_tok_vals)[0, 1]
corr_old = np.corrcoef(old_ld_vals, n_tok_vals)[0, 1]

print(f"Correlacion con longitud -- lexical_diversity ORIGINAL: {corr_old:.4f}")
print(f"Correlacion con longitud -- MATTR (nueva):              {corr_mattr:.4f}")
print(f"\nReduccion de la dependencia de longitud: {abs(corr_old) - abs(corr_mattr):.4f} puntos de correlacion")


Correlacion con longitud -- lexical_diversity ORIGINAL: -0.8074
Correlacion con longitud -- MATTR (nueva):              -0.1696

Reduccion de la dependencia de longitud: 0.6378 puntos de correlacion


## Pruebas unitarias (las 4 características)

In [6]:
def test_lex_pol_neg_invierte_negacion():
    tokens = tt.tokenize(TextProcessing.transformer("no me gusta nada esto"))
    assert FeatureExtraction.lex_pol_neg(tokens) < 0
    tokens2 = tt.tokenize(TextProcessing.transformer("me encanta esto"))
    assert FeatureExtraction.lex_pol_neg(tokens2) > 0
    print("OK: lex_pol_neg invierte polaridad bajo negacion")

def test_elongated_detecta_repeticion():
    assert FeatureExtraction.elongated_words_count("buenooo dia") == 1.0
    assert FeatureExtraction.elongated_words_count("hola amigo") == 0.0
    print("OK: elongated_words_count detecta elongacion real")

def test_punct_repetition_detecta_signos():
    assert FeatureExtraction.punct_repetition("que bien!!!") == 1.0
    assert FeatureExtraction.punct_repetition("hola.") == 0.0
    print("OK: punct_repetition detecta signos repetidos")

def test_mattr_en_rango_valido():
    corto = tt.tokenize(TextProcessing.transformer("hola amigo"))
    largo = tt.tokenize(TextProcessing.transformer(
        "hola amigo como estas hoy espero que todo bien por alla"))
    v_corto = FeatureExtraction.lexical_diversity_mattr(corto, window=5)
    v_largo = FeatureExtraction.lexical_diversity_mattr(largo, window=5)
    assert 0 <= v_corto <= 1 and 0 <= v_largo <= 1
    print(f"OK: MATTR en rango [0,1] -- corto={v_corto}, largo={v_largo}")

test_lex_pol_neg_invierte_negacion()
test_elongated_detecta_repeticion()
test_punct_repetition_detecta_signos()
test_mattr_en_rango_valido()


OK: lex_pol_neg invierte polaridad bajo negacion
OK: elongated_words_count detecta elongacion real
OK: punct_repetition detecta signos repetidos
OK: MATTR en rango [0,1] -- corto=1.0, largo=1.0


## Cobertura sobre el corpus completo

Cuántos tweets activan cada característica nueva (evidencia de que no son constantes):


In [7]:
activaciones = {'lex_pol_neg_no_cero': 0, 'elongacion': 0, 'punct_rep': 0}
total = 0

for t in texts:
    proc = TextProcessing.transformer(t)
    if not proc:
        continue
    tokens = tt.tokenize(proc)
    total += 1
    if FeatureExtraction.lex_pol_neg(tokens) != 0:
        activaciones['lex_pol_neg_no_cero'] += 1
    if FeatureExtraction.elongated_words_count(t) > 0:
        activaciones['elongacion'] += 1
    if FeatureExtraction.punct_repetition(t) > 0:
        activaciones['punct_rep'] += 1

for k, v in activaciones.items():
    print(f"{k}: {v} de {total} tweets ({100*v/total:.1f}%)")


lex_pol_neg_no_cero: 267 de 1008 tweets (26.5%)
elongacion: 30 de 1008 tweets (3.0%)
punct_rep: 49 de 1008 tweets (4.9%)
